[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/01_fundamentals/tensors.ipynb)

# 01 . Tensors

## Concept — Tensors (revision notes)

**What a tensor is**
- A multi-dimensional array, like a NumPy array, but with two extra abilities:
  it can live on a GPU, and it can automatically track gradients.
- Every single thing in PyTorch is a tensor: your input data, your model's
  weights, the activations flowing through the network, the gradients, the loss.

**Why tensors exist (not just NumPy arrays)**
- Deep learning = huge amounts of matrix multiplication + elementwise math,
  repeated millions of times, ideally in parallel.
- NumPy arrays cannot run on a GPU.
- NumPy arrays cannot track gradients for backpropagation.
- Tensors solve both problems while keeping a NumPy-like API.

**What a tensor actually holds (the 3-part mental model)**
- **Data** — the raw numbers, sitting in one flat block of memory.
- **Shape / strides** — metadata that says how to read that flat block as an
  N-dimensional array. (This is *why* operations like `.view()` are free —
  same underlying data, just different metadata describing its shape.)
- **Device** — where the data physically lives: CPU RAM or GPU VRAM.
  Operations between tensors on different devices raise an error — you must
  move one of them first.

**dtype — the most common source of silent bugs**
- `float32` is the safe default for almost everything.
- `torch.tensor([1, 2, 3])` -> infers `int64` (whole numbers -> integer type).
- `torch.tensor([1.0, 2.0, 3.0])` -> infers `float32` (has decimals -> float type).
- NumPy defaults to `float64`, PyTorch defaults to `float32` — mixing the two
  libraries without casting is a classic bug.
- Cast deliberately with `.float()`, `.double()`, `.long()`, or `.to(dtype)`.

**Views vs copies — memory sharing**
- `.view()`, `.reshape()` (usually), slicing (`a[:, 0]`), and `.T` do **not**
  copy data. They return a new tensor object that **shares** the same
  underlying memory as the original.
- Modifying a view modifies the original too — surprising if you don't expect it.
- `.clone()` makes a true, independent copy.
- Rule of thumb: if you're not sure, check `.data_ptr()` — same pointer value
  means shared memory.

**In-place operations**
- Any method ending in `_` modifies the tensor directly instead of returning
  a new one: `x.add_(1)` changes `x`; `x.add(1)` returns a new tensor, `x` is untouched.
- In-place ops save memory (no new allocation) but can silently break
  autograd if the original (un-modified) value was needed for a backward pass.
- Safe rule for beginners: avoid in-place ops on any tensor that requires
  gradients, unless you specifically understand why it's safe in that spot.

**Broadcasting**
- Lets you do elementwise ops between tensors of *different but compatible*
  shapes, without manually expanding either one.
- Rule: align shapes starting from the rightmost dimension; a dimension of
  size 1 is stretched to match the other tensor's size in that dimension;
  dimensions must either match exactly or one of them must be 1.
- Example: shape `(3, 1)` and shape `(1, 4)` broadcast together into `(3, 4)`.

**Reductions and dimensions (`dim` argument)**
- Operations like `.sum()`, `.mean()`, `.max()` take a `dim` argument saying
  *which* dimension to collapse.
- `dim=0` collapses rows (operates "down" each column) -> typically used to
  reduce across a batch.
- `dim=1` collapses columns (operates "across" each row) -> typically used to
  reduce across features.
- `keepdim=True` keeps the collapsed dimension as size 1 instead of removing
  it entirely — very useful for broadcasting the result back against the original.

**Common tensor creation functions, quick reference**
- `torch.zeros(shape)`, `torch.ones(shape)` — filled with constants.
- `torch.rand(shape)` — uniform random in `[0, 1)`.
- `torch.randn(shape)` — standard normal random (mean 0, std 1).
- `torch.arange(start, end, step)` — like Python's `range`, but a tensor.
- `torch.linspace(start, end, steps)` — `steps` evenly spaced points between
  `start` and `end` (inclusive).
- `torch.zeros_like(x)`, `torch.ones_like(x)`, `torch.randn_like(x)` — same
  shape/dtype/device as an existing tensor `x`.
- `torch.eye(n)` — identity matrix.


### 1. Creating tensors — the basics

**What this cell does (plain language):** we make tensors a few different
ways — from a plain Python list, filled with zeros/ones, filled with random
numbers, and as a numbered range — and print each one's shape and dtype so
you can see exactly what got created.


In [1]:
import torch
import numpy as np

# from a nested Python list -> PyTorch infers shape and dtype automatically
t_from_list = torch.tensor([[1, 2], [3, 4]])
print("from list:\n", t_from_list)
print("shape:", t_from_list.shape, "| dtype:", t_from_list.dtype)

# filled with a constant
t_zeros = torch.zeros(2, 3)
t_ones = torch.ones(2, 3)
print("\nzeros:\n", t_zeros)
print("ones:\n", t_ones)

# random tensors - two different distributions, easy to mix up
t_rand = torch.rand(2, 3)   # uniform in [0, 1)
t_randn = torch.randn(2, 3) # standard normal (can be negative, centered at 0)
print("\nuniform rand (all in [0,1)):\n", t_rand)
print("normal randn (centered at 0, can be negative):\n", t_randn)

# a numbered sequence, like Python's range() but as a tensor
t_arange = torch.arange(0, 10, 2)
print("\narange(0, 10, 2):", t_arange)

# evenly spaced points including both endpoints - different from arange!
t_linspace = torch.linspace(0, 1, 5)
print("linspace(0, 1, 5):", t_linspace)

# same shape/dtype/device as an existing tensor - very common in real code
t_like = torch.zeros_like(t_rand)
print("\nzeros_like(t_rand) shape:", t_like.shape, "dtype:", t_like.dtype)


from list:
 tensor([[1, 2],
        [3, 4]])
shape: torch.Size([2, 2]) | dtype: torch.int64

zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])

uniform rand (all in [0,1)):
 tensor([[0.6745, 0.2080, 0.8421],
        [0.8007, 0.5072, 0.3446]])
normal randn (centered at 0, can be negative):
 tensor([[ 0.1245, -0.0243, -1.4924],
        [ 0.5158,  0.0064, -1.6350]])

arange(0, 10, 2): tensor([0, 2, 4, 6, 8])
linspace(0, 1, 5): tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])

zeros_like(t_rand) shape: torch.Size([2, 3]) dtype: torch.float32


### 2. Dtypes — where silent bugs come from

**What this cell does (plain language):** we create tensors from whole
numbers vs decimal numbers and show PyTorch picks a different dtype for
each automatically. Then we deliberately cast between dtypes so you see the
difference between letting PyTorch guess and doing it on purpose.


In [ ]:
int_tensor = torch.tensor([1, 2, 3])          # whole numbers -> infers int64
float_tensor = torch.tensor([1.0, 2.0, 3.0])  # has a decimal point -> infers float32
print("int_tensor dtype:", int_tensor.dtype)
print("float_tensor dtype:", float_tensor.dtype)

# Explicit casting - always do this on purpose, don't let it happen by accident
casted_to_float = int_tensor.to(torch.float32)
casted_to_long = float_tensor.to(torch.long)  # long == int64, common for labels/indices
print("\nint_tensor cast to float32:", casted_to_float, casted_to_float.dtype)
print("float_tensor cast to long:", casted_to_long, casted_to_long.dtype)

# true division automatically promotes int -> float (this one is safe/expected)
result = int_tensor / 2
print("\nint_tensor / 2 =", result, "| dtype:", result.dtype)

# but this is the classic real-world trap: NumPy defaults to float64
np_array = np.array([1.0, 2.0, 3.0])
torch_from_np = torch.from_numpy(np_array)
print("\nNumPy array dtype:", np_array.dtype)
print("Tensor made from it:", torch_from_np.dtype, "<- NOT float32! this will clash")
print("with a float32 model unless you cast it: .float()")


int_tensor dtype: torch.int64
float_tensor dtype: torch.float32

int_tensor cast to float32: tensor([1., 2., 3.]) torch.float32
float_tensor cast to long: tensor([1, 2, 3]) torch.int64

int_tensor / 2 = tensor([0.5000, 1.0000, 1.5000]) | dtype: torch.float32

NumPy array dtype: float64
Tensor made from it: torch.float64 <- NOT float32! this will clash
with a float32 model unless you cast it: .float()


### 3. Devices — CPU vs GPU

**What this cell does (plain language):** we check whether a GPU is
available, create a tensor on CPU, then move a copy of it to whichever
device we found. If you're running this on a machine without a GPU, it will
correctly report CPU and everything still works the same way logically.


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Detected device:", device)

cpu_tensor = torch.randn(2, 2)
moved_tensor = cpu_tensor.to(device)  # .to() returns a NEW tensor on that device
print("cpu_tensor lives on:", cpu_tensor.device)
print("moved_tensor lives on:", moved_tensor.device)

# IMPORTANT: .to(device) does not modify cpu_tensor in place - it returns a
# new tensor. cpu_tensor itself is still on the CPU after this line.
print("\ncpu_tensor device is STILL:", cpu_tensor.device, "(unchanged - .to() is not in-place)")

# Ops between tensors on different devices raise a RuntimeError.
# Uncomment the next two lines on a machine with a GPU to see it fail:
# gpu_only_tensor = torch.randn(2, 2).cuda()
# gpu_only_tensor + cpu_tensor  # RuntimeError: expected all tensors on same device


Detected device: cuda
cpu_tensor lives on: cpu
moved_tensor lives on: cuda:0

cpu_tensor device is STILL: cpu (unchanged - .to() is not in-place)


### 4. Views vs copies — the shared-memory gotcha

**What this cell does (plain language):** we create a tensor, make a "view"
of it with `.view()`, and a real independent "copy" of it with `.clone()`.
Then we modify the view and the copy separately and watch what happens to
the original — this is the exact behavior that causes confusing bugs if you
don't expect it.


In [4]:
original = torch.arange(6)
print("original:", original)

view = original.view(2, 3)              # SHARES memory with original
copy = original.clone().view(2, 3)      # INDEPENDENT memory

print("\nview:\n", view)
print("copy:\n", copy)

view[0, 0] = 999
print("\nafter setting view[0,0] = 999:")
print("original is now:", original, "<- changed! view shares memory with it")

copy[0, 0] = -1
print("\nafter setting copy[0,0] = -1:")
print("original is STILL:", original, "<- unaffected, copy was independent")

# Quick way to check if two tensors share memory: compare data_ptr()
print("\noriginal.data_ptr() == view.data_ptr():", original.data_ptr() == view.data_ptr())
print("original.data_ptr() == copy.data_ptr():", original.data_ptr() == copy.data_ptr())


original: tensor([0, 1, 2, 3, 4, 5])

view:
 tensor([[0, 1, 2],
        [3, 4, 5]])
copy:
 tensor([[0, 1, 2],
        [3, 4, 5]])

after setting view[0,0] = 999:
original is now: tensor([999,   1,   2,   3,   4,   5]) <- changed! view shares memory with it

after setting copy[0,0] = -1:
original is STILL: tensor([999,   1,   2,   3,   4,   5]) <- unaffected, copy was independent

original.data_ptr() == view.data_ptr(): True
original.data_ptr() == copy.data_ptr(): False


### 5. In-place operations (methods ending in `_`)

**What this cell does (plain language):** we compare a normal operation
(`.add()`, returns something new) against its in-place twin (`.add_()`,
changes the tensor directly) so you can see exactly what "in-place" means in
practice.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])

y = x.add(1)       # NOT in-place: returns a brand new tensor
print("x after x.add(1):", x, "  (unchanged)")
print("y (the new tensor):", y)

x.add_(1)          # IS in-place: modifies x directly, note the trailing underscore
print("\nx after x.add_(1):", x, "  (changed!)")

# Same pattern applies to almost every op: relu/relu_, mul/mul_, clamp/clamp_, etc.
z = torch.tensor([-1.0, 0.5, 2.0])
print("\nz.relu() (not in-place):", z.relu(), "| z is still:", z)
z.relu_()
print("z.relu_() (in-place):", z)


x after x.add(1): tensor([1., 2., 3.])   (unchanged)
y (the new tensor): tensor([2., 3., 4.])

x after x.add_(1): tensor([2., 3., 4.])   (changed!)

z.relu() (not in-place): tensor([0.0000, 0.5000, 2.0000]) | z is still: tensor([-1.0000,  0.5000,  2.0000])
z.relu_() (in-place): tensor([0.0000, 0.5000, 2.0000])


### 6. Broadcasting — operating on different shapes

**What this cell does (plain language):** we add a `(3, 1)` tensor to a
`(1, 4)` tensor. Neither shape matches the other, but broadcasting rules let
PyTorch stretch each one to a common `(3, 4)` shape automatically, without
you writing any loop or manual expansion.


In [6]:
a = torch.ones(3, 1)       # shape (3, 1)
b = torch.ones(1, 4) * 2   # shape (1, 4)

c = a + b  # broadcasts: dimension of size 1 stretches to match the other tensor
print("a shape:", a.shape, "| b shape:", b.shape, "| result shape:", c.shape)
print(c)

# A shape that CANNOT broadcast will raise an error - dimensions must match
# or be 1, there's no other rule
try:
    bad = torch.ones(3, 2) + torch.ones(4, 2)  # 3 vs 4 in dim 0, neither is 1 -> error
except RuntimeError as e:
    print("\nExpected broadcasting error:", str(e)[:100])


a shape: torch.Size([3, 1]) | b shape: torch.Size([1, 4]) | result shape: torch.Size([3, 4])
tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]])

Expected broadcasting error: The size of tensor a (3) must match the size of tensor b (4) at non-singleton dimension 0


### 7. Reductions and the `dim` argument

**What this cell does (plain language):** we build a small 2D tensor shaped
like a mini-batch (3 rows = samples, 4 columns = features) and reduce it
along each dimension separately, printing the result so you can see exactly
what `dim=0` vs `dim=1` each collapse.


In [ ]:
batch = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [5.0, 6.0, 7.0, 8.0],
    [9.0, 10.0, 11.0, 12.0],
])
print("batch shape (3 samples, 4 features):", batch.shape)

# dim=0 collapses ACROSS samples (down each column) -> one value per feature
mean_per_feature = batch.mean(dim=0)
print("\nmean(dim=0) - one value per feature:", mean_per_feature, "shape:", mean_per_feature.shape)

# dim=1 collapses ACROSS features (across each row) -> one value per sample
mean_per_sample = batch.mean(dim=1)
print("mean(dim=1) - one value per sample:", mean_per_sample, "shape:", mean_per_sample.shape)

# keepdim=True keeps the reduced dimension as size 1 instead of dropping it
# useful when you want to broadcast the result back against the original tensor
mean_keepdim = batch.mean(dim=1, keepdim=True)
print("\nmean(dim=1, keepdim=True) shape:", mean_keepdim.shape, "(vs", mean_per_sample.shape, "without keepdim)")

centered = batch - mean_keepdim  # broadcasts (3,1) against (3,4) correctly
print("\nbatch centered (each row minus its own mean):\n", centered)


batch shape (3 samples, 4 features): torch.Size([3, 4])

mean(dim=0) - one value per feature: tensor([5., 6., 7., 8.]) shape: torch.Size([4])
mean(dim=1) - one value per sample: tensor([ 2.5000,  6.5000, 10.5000]) shape: torch.Size([3])

mean(dim=1, keepdim=True) shape: torch.Size([3, 1]) (vs torch.Size([3]) without keepdim)

batch centered (each row minus its own mean):
 tensor([[-1.5000, -0.5000,  0.5000,  1.5000],
        [-1.5000, -0.5000,  0.5000,  1.5000],
        [-1.5000, -0.5000,  0.5000,  1.5000]])


### 8. Indexing and slicing

**What this cell does (plain language):** we slice a 2D tensor several
different ways — a single row, a single column, a sub-block — the same
slicing syntax you already know from Python lists and NumPy.


In [ ]:
m = torch.arange(12).reshape(3, 4)
print("m:\n", m)

print("\nfirst row m[0]:", m[0])
print("first column m[:, 0]:", m[:, 0])
print("sub-block m[0:2, 1:3]:\n", m[0:2, 1:3])
print("last row m[-1]:", m[-1])

# Boolean masking - select elements matching a condition
mask = m > 6
print("\nmask (m > 6):\n", mask)
print("m[mask] (flattened, only matching elements):", m[mask])

# fancy indexing with a list of indices
print("\nm[:, [0, 2]] (columns 0 and 2):\n", m[:, [0, 2]])


m:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

first row m[0]: tensor([0, 1, 2, 3])
first column m[:, 0]: tensor([0, 4, 8])
sub-block m[0:2, 1:3]:
 tensor([[1, 2],
        [5, 6]])
last row m[-1]: tensor([ 8,  9, 10, 11])

mask (m > 6):
 tensor([[False, False, False, False],
        [False, False, False,  True],
        [ True,  True,  True,  True]])
m[mask] (flattened, only matching elements): tensor([ 7,  8,  9, 10, 11])

m[:, [0, 2]] (columns 0 and 2):
 tensor([[ 0,  2],
        [ 4,  6],
        [ 8, 10]])


### 9. NumPy interoperability

**What this cell does (plain language):** we convert a NumPy array to a
tensor and back, and show that `torch.from_numpy()` shares memory with the
original array — modifying one changes the other, same shared-memory idea as views.


In [9]:
np_array = np.array([1, 2, 3])
from_np = torch.from_numpy(np_array)  # SHARES memory with np_array
print("from_np:", from_np)

np_array[0] = 999
print("after modifying np_array, from_np is now:", from_np, "<- also changed!")

# going back to numpy - only works for CPU tensors
back_to_np = from_np.numpy()
print("\nback_to_np:", back_to_np, "type:", type(back_to_np))

# if you want an independent copy instead of shared memory, use torch.tensor()
independent = torch.tensor(np_array)  # COPIES the data
np_array[0] = -1
print("\nafter modifying np_array again, independent tensor is still:", independent)


from_np: tensor([1, 2, 3])
after modifying np_array, from_np is now: tensor([999,   2,   3]) <- also changed!

back_to_np: [999   2   3] type: <class 'numpy.ndarray'>

after modifying np_array again, independent tensor is still: tensor([999,   2,   3])


### 10. Reshaping: view vs reshape vs flatten

**What this cell does (plain language):** we compare the three main ways
to change a tensor's shape and show when `.view()` fails but `.reshape()`
succeeds anyway (because it falls back to copying when the data isn't
laid out in a way that allows a plain view).


In [ ]:
x = torch.arange(12)
print("original:", x, "shape:", x.shape)

v = x.view(3, 4)          # requires the data to be "contiguous" in memory
r = x.reshape(3, 4)       # tries view first, copies if it must
f = x.view(3, 4).flatten()  # flatten back to 1D
print("\nview(3,4):\n", v)
print("reshape(3,4):\n", r)
print("flatten():", f)

# The classic view failure: after a transpose, the data is no longer
# contiguous in memory, so .view() fails but .reshape() still works
transposed = v.t()  # transpose - this creates a non-contiguous view
print("\ntransposed.is_contiguous():", transposed.is_contiguous())
try:
    transposed.view(12)  # fails!
except RuntimeError as e:
    print("view() error on non-contiguous tensor:", str(e)[:100])

# reshape() handles it by copying automatically when needed
works = transposed.reshape(12)
print("reshape() on the same tensor works fine:", works)


original: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]) shape: torch.Size([12])

view(3,4):
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
reshape(3,4):
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
flatten(): tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

transposed.is_contiguous(): False
view() error on non-contiguous tensor: view size is not compatible with input tensor's size and stride (at least one dimension spans across
reshape() on the same tensor works fine: tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11])


## Common bugs

- **`RuntimeError: expected scalar type Float but found Double`**
  Mixed `float32` and `float64` — very often because NumPy defaults to
  `float64` while PyTorch defaults to `float32`. Fix: cast explicitly with
  `.float()` or `.double()` right when you convert from NumPy.

- **`RuntimeError: Expected all tensors to be on the same device`**
  Model is on GPU but the input batch is still on CPU (or vice versa) — very
  common right after `.to(device)`ing a model but forgetting the data.
  Fix: move *every* tensor coming out of the DataLoader, every batch:
  `inputs = inputs.to(device)`.

- **Silent wrong results from views**
  You modified what you thought was an independent copy, but it was actually
  a view sharing memory with the original. Fix: use `.clone()` whenever you
  need a real independent copy; check `.data_ptr()` if you're unsure.

- **`a leaf Variable that requires grad is being used in an in-place operation`**
  Used an in-place op (`x.add_(1)`) on a tensor that autograd needed
  unmodified for the backward pass. Fix: avoid in-place ops on tensors that
  require gradients unless you specifically know it's safe there.

- **`.numpy()` fails with "can't convert cuda:0 device type tensor to numpy"**
  Tried to call `.numpy()` directly on a GPU tensor — NumPy only understands
  CPU memory. Fix: `tensor.detach().cpu().numpy()`.

- **Broadcasting silently produces the wrong shape**
  An unexpected size-1 dimension got broadcast in a way you didn't intend,
  instead of raising an error. Fix: print `.shape` before and after any op
  you're not 100% sure about; use `.unsqueeze()`/`.squeeze()` explicitly when
  in doubt instead of relying on broadcasting to guess your intent.

- **`.view()` fails with "input is not contiguous"**
  Happened after a transpose/permute — the data is no longer laid out
  contiguously in memory. Fix: use `.reshape()` instead (it copies
  automatically when needed), or call `.contiguous()` before `.view()`.

- **Forgetting `dim` in a reduction gives an unexpected scalar**
  `tensor.sum()` with no `dim` argument reduces over *all* dimensions down
  to a single number, which surprises people expecting a per-row or
  per-column result. Fix: always pass `dim=` explicitly when you want a
  reduction along one axis specifically.


## Exercises

Attempt each one in the empty cell right after the question. My full solution is in the cell after that - don't peek until you've tried.

**Exercise 1 - Predict then verify the dtype.**
What is the dtype of the result of
`torch.tensor([1, 2, 3]) + torch.tensor([1.0, 2.0, 3.0])`? Write down your
prediction, then write code to check it.

In [11]:
# Your attempt for Exercise 1 here\n

**Solution 1**

In [ ]:
# Solution 1
a = torch.tensor([1, 2, 3])          # int64
b = torch.tensor([1.0, 2.0, 3.0])    # float32
result = a + b
print("dtype:", result.dtype)  # float32 - PyTorch promotes int to float automatically
# Prediction: since one operand is float32, the whole result is promoted to float32.


dtype: torch.float32


**Exercise 2 - View or copy?**
Given `a = torch.arange(12).reshape(3, 4)`, is `b = a[:, 1]` a view or a
copy? Write code that proves your answer by modifying `b[0]` and checking
whether `a` changed.

In [ ]:
# Your attempt for Exercise 2 here\n

**Solution 2**

In [ ]:
# Solution 2
a = torch.arange(12).reshape(3, 4)
b = a[:, 1]  # slicing -> this is a VIEW, shares memory with a
print("before:", a)
b[0] = 999
print("after modifying b[0]:", a)  # a[0, 1] changed too -> confirms it's a view


before: tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
after modifying b[0]: tensor([[  0, 999,   2,   3],
        [  4,   5,   6,   7],
        [  8,   9,  10,  11]])


**Exercise 3 - Broadcasting shapes.**
For each pair of shapes below, write code that creates tensors of those
shapes, adds them, and prints the resulting shape (or catches the error if
they can't broadcast):
`(3,4)` & `(4,)` , `(3,1)` & `(1,4)` , `(3,4)` & `(3,)` , `(2,3,4)` & `(3,4)`

In [ ]:
# Your attempt for Exercise 3 here\n

**Solution 3**

In [16]:
# Solution 3
import torch

def try_broadcast(shape_a, shape_b):
    a = torch.ones(shape_a)
    b = torch.ones(shape_b)
    try:
        c = a + b
        print(f"{shape_a} + {shape_b} -> broadcasts to {tuple(c.shape)}")
    except RuntimeError as e:
        print(f"{shape_a} + {shape_b} -> ERROR: cannot broadcast")

try_broadcast((3, 4), (4,))
try_broadcast((3, 1), (1, 4))
try_broadcast((3, 4), (3,))   # this one actually FAILS - (3,) aligns to last dim (4), mismatch
try_broadcast((2, 3, 4), (3, 4))


(3, 4) + (4,) -> broadcasts to (3, 4)
(3, 1) + (1, 4) -> broadcasts to (3, 4)
(3, 4) + (3,) -> ERROR: cannot broadcast
(2, 3, 4) + (3, 4) -> broadcasts to (2, 3, 4)


**Exercise 4 - Fix the device bug.**
The following code raises a `RuntimeError`. Explain why, then rewrite it so
it works (assume `SomeModel` is any `nn.Module` and a GPU may or may not be
present):
```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SomeModel().to(device)
x = torch.randn(8, 10)
output = model(x)
```

In [ ]:
# Your attempt for Exercise 4 here\n

**Solution 4**

In [18]:
# Solution 4
# The bug: `x` is created on the default device (CPU) but never moved to
# match the model, which was moved with .to(device). If device is "cuda",
# this raises: RuntimeError: Expected all tensors to be on the same device.
# Fix: move the input tensor to the same device as the model.

import torch
import torch.nn as nn

class SomeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 1)
    def forward(self, x):
        return self.linear(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SomeModel().to(device)
x = torch.randn(8, 10).to(device)  # <- the fix: move the input too
output = model(x)
print("output shape:", output.shape, "| device:", output.device)


output shape: torch.Size([8, 1]) | device: cuda:0


**Exercise 5 - Write `normalize(x)`.**
Write a function `normalize(x)` that takes a tensor of shape
`(batch, features)` and returns a new tensor with each *row* scaled to zero
mean and unit variance. Use only tensor ops - no Python loops. (Hint: you'll
need `dim` and `keepdim`.)

In [19]:
# Your attempt for Exercise 5 here\n

**Solution 5**

In [ ]:
# Solution 5
def normalize(x):
    mean = x.mean(dim=1, keepdim=True)   # one mean per row
    std = x.std(dim=1, keepdim=True)     # one std per row
    return (x - mean) / (std + 1e-8)     # small epsilon avoids divide-by-zero

x = torch.tensor([[1.0, 2.0, 3.0], [10.0, 20.0, 30.0]])
normalized = normalize(x)
print("normalized:\n", normalized)
print("per-row mean (should be ~0):", normalized.mean(dim=1))
print("per-row std (should be ~1):", normalized.std(dim=1))


normalized:
 tensor([[-1.,  0.,  1.],
        [-1.,  0.,  1.]])
per-row mean (should be ~0): tensor([0., 0.])
per-row std (should be ~1): tensor([1., 1.])


**Exercise 6 - In-place danger.**
Write a snippet where a tensor `y` is used twice in a computation (so
autograd needs to remember its value), then modified in-place afterward —
this should raise a `RuntimeError` on `.backward()`. Then write a safe
version using the non-in-place equivalent. Explain the difference in a comment.

In [ ]:
# Your attempt for Exercise 6 here\n

**Solution 6**

In [22]:
# Solution 6
import torch

# UNSAFE example: y is used TWICE in y * y, so autograd saves y's value to
# compute the backward pass. Modifying y in-place afterward corrupts that
# saved value, and backward() raises a RuntimeError.
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x * 2
z = y * y          # z depends on y TWICE -> autograd must remember y's value
try:
    y.add_(1)      # in-place op on y AFTER it was saved for z's backward
    loss = z.sum()
    loss.backward()
    print("unsafe case did not error (unexpected)")
except RuntimeError as e:
    print("unsafe case failed as expected:", str(e)[:120])

# SAFE example: use the non-in-place version instead - a new tensor is
# created, so nothing already-saved gets overwritten
a = torch.tensor([1.0, 2.0], requires_grad=True)
b = a * 2
c = b.add(1)       # non-in-place - creates a new tensor, b is untouched
d = b * b          # still safe: b itself was never modified
loss2 = (c.sum() + d.sum())
loss2.backward()
print("safe case grad:", a.grad)


unsafe case failed as expected: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor [2]],
safe case grad: tensor([10., 18.])


**Exercise 7 - Contiguous vs non-contiguous.**
Create a `(4, 4)` tensor, transpose it, and try `.view(16)` on the result.
It should fail. Show the fix using `.reshape()` and, separately, using
`.contiguous().view(16)`.

In [23]:
# Your attempt for Exercise 7 here\n

**Solution 7**

In [ ]:
# Solution 7
x = torch.arange(16).reshape(4, 4)
transposed = x.t()
print("is_contiguous:", transposed.is_contiguous())

try:
    transposed.view(16)
except RuntimeError as e:
    print("view() failed as expected:", str(e)[:80])

# Fix option 1: reshape() copies automatically when needed
fixed_reshape = transposed.reshape(16)
print("reshape() works:", fixed_reshape)

# Fix option 2: force contiguous memory first, then view() works fine
fixed_contiguous = transposed.contiguous().view(16)
print("contiguous().view() works:", fixed_contiguous)


is_contiguous: False
view() failed as expected: view size is not compatible with input tensor's size and stride (at least one di
reshape() works: tensor([ 0,  4,  8, 12,  1,  5,  9, 13,  2,  6, 10, 14,  3,  7, 11, 15])
contiguous().view() works: tensor([ 0,  4,  8, 12,  1,  5,  9, 13,  2,  6, 10, 14,  3,  7, 11, 15])


**Exercise 8 - Boolean masking.**
Given `m = torch.arange(20).reshape(4, 5)`, write code that replaces every
even number in `m` with `-1`, without using a Python loop.

In [25]:
# Your attempt for Exercise 8 here\n

**Solution 8**

In [ ]:
# Solution 8
m = torch.arange(20).reshape(4, 5)
print("before:\n", m)

mask = (m % 2 == 0)
m[mask] = -1
print("after replacing evens with -1:\n", m)


before:
 tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]])
after replacing evens with -1:
 tensor([[-1,  1, -1,  3, -1],
        [ 5, -1,  7, -1,  9],
        [-1, 11, -1, 13, -1],
        [15, -1, 17, -1, 19]])


**Exercise 9 - NumPy round-trip safety.**
Write code that converts a NumPy array to a tensor in a way that does
**NOT** share memory with the original array (i.e. modifying the NumPy
array afterward should NOT change the tensor). Prove it works.

In [ ]:
# Your attempt for Exercise 9 here\n

**Solution 9**

In [ ]:
# Solution 9
import numpy as np

np_array = np.array([1, 2, 3])

# torch.tensor() COPIES data, unlike torch.from_numpy() which shares memory
independent = torch.tensor(np_array)

np_array[0] = 999
print("np_array after modification:", np_array)
print("independent tensor (should be unchanged):", independent)


np_array after modification: [999   2   3]
independent tensor (should be unchanged): tensor([1, 2, 3])


**Exercise 10 - Per-sample vs per-feature reduction.**
Given a `(5, 3)` tensor representing 5 samples with 3 features each, compute
(a) the max value per sample, and (b) the max value per feature - using
`dim` correctly for each - and print both results with their shapes.

In [29]:
# Your attempt for Exercise 10 here\n

**Solution 10**

In [ ]:
# Solution 10
data = torch.randn(5, 3)
print("data:\n", data)

max_per_sample = data.max(dim=1).values   # collapse across features -> one value per sample
print("\nmax per sample:", max_per_sample, "shape:", max_per_sample.shape)

max_per_feature = data.max(dim=0).values  # collapse across samples -> one value per feature
print("max per feature:", max_per_feature, "shape:", max_per_feature.shape)


data:
 tensor([[ 0.8286,  0.8583,  1.2774],
        [-0.1075, -0.2112, -0.2983],
        [ 0.6821,  0.1755,  0.6708],
        [ 0.0570,  0.7024, -1.3547],
        [-0.6896,  0.1705,  0.5123]])

max per sample: tensor([ 1.2774, -0.1075,  0.6821,  0.7024,  0.5123]) shape: torch.Size([5])
max per feature: tensor([0.8286, 0.8583, 1.2774]) shape: torch.Size([3])
